# Run all evals across the 5 conditions

Loops over the 5 model specs (base no-sys, base + C-3PO system prompt, FT-demos, FT-first-person, FT-SDF) calling `evals.runner.run` per spec, then aggregates a 5×3 summary CSV. Designed for Colab A100. FT specs are skipped automatically until adapters exist.

In [ ]:
import os, sys
if 'COLAB_RELEASE_TAG' in os.environ:
    !pip install -q transformers accelerate peft pydantic pandas matplotlib localrouter
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['HF_TOKEN'] = userdata.get('hf_token')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('clr_openrouter')
    REPO = '/content/drive/MyDrive/clr_worktest'
    if not os.path.isdir(REPO):
        !git clone https://github.com/Junekhunter/clr-worktest.git $REPO
    %cd $REPO
    !git pull
else:
    %cd /home/hunter/ai/clr_worktest

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from evals import runner, aggregate
from evals.model import ModelSpec

EVALS_DIR   = Path('evals')
RESULTS_DIR = Path('/content/drive/MyDrive/clr_worktest_results') if 'COLAB_RELEASE_TAG' in os.environ else Path('results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
C3PO_SYS = json.loads(Path('configs/base_c3po_sys.json').read_text())['system_prompt']

SPECS = [
    ModelSpec(model_id='base_no_sys'),
    ModelSpec(model_id='base_c3po_sys', system_prompt=C3PO_SYS),
    # Fill these in once adapters are trained:
    # ModelSpec(model_id='ft_demos',         adapter='Junekhunter/clr-c3po-demos'),
    # ModelSpec(model_id='ft_first_person',  adapter='Junekhunter/clr-c3po-firstperson'),
    # ModelSpec(model_id='ft_sdf',           adapter='Junekhunter/clr-c3po-sdf'),
]
len(SPECS)

## Smoke (per CLAUDE.md staged pipeline) — first 2 prompts/questions per eval, base only

In [ ]:
# Make a tiny eval set on disk for the smoke pass.
smoke_dir = Path('evals_smoke'); (smoke_dir / 'prompts').mkdir(parents=True, exist_ok=True)
for f in ['identity_prompts.json', 'behavioral_prompts.json', 'factual_questions.json']:
    src = json.loads((EVALS_DIR / 'prompts' / f).read_text())
    key = 'questions' if 'questions' in src else 'prompts'
    src[key] = src[key][:2]
    (smoke_dir / 'prompts' / f).write_text(json.dumps(src, indent=2))
(smoke_dir / 'judge_prompt.md').write_text((EVALS_DIR / 'judge_prompt.md').read_text())

runner.run(SPECS[0], smoke_dir, RESULTS_DIR / 'smoke', force=True)
print('smoke OK')

## Full run across all configured specs

In [ ]:
for spec in SPECS:
    runner.run(spec, EVALS_DIR, RESULTS_DIR)

In [ ]:
rows = aggregate.main(RESULTS_DIR, RESULTS_DIR / 'summary.csv')
df = pd.read_csv(RESULTS_DIR / 'summary.csv')
df

In [ ]:
# Slide-friendly bar charts with 95% CI error bars.
import numpy as np

def _bars(ax, df, mean_col, lo_col, hi_col, title, ylabel):
    m = df[mean_col].astype(float)
    lo = df[lo_col].astype(float); hi = df[hi_col].astype(float)
    err = np.vstack([m - lo, hi - m])
    ax.bar(df['model_id'], m, yerr=err, capsize=4)
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=30)

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
_bars(axes[0,0], df, 'identity_score',       'identity_ci_lo',       'identity_ci_hi',       'Identity (P_c3po − P_generic)', 'score')
_bars(axes[0,1], df, 'factual_trained_acc',  'factual_trained_ci_lo','factual_trained_ci_hi','Factual MC — trained', 'accuracy')
_bars(axes[0,2], df, 'factual_held_out_acc', 'factual_held_out_ci_lo','factual_held_out_ci_hi','Factual MC — held-out', 'accuracy')
axes[0,3].axis('off')
_bars(axes[1,0], df, 'bh_formality',  'bh_formality_ci_lo',  'bh_formality_ci_hi',  'Behavioral: formality (1-5)',  'mean')
_bars(axes[1,1], df, 'bh_verbosity',  'bh_verbosity_ci_lo',  'bh_verbosity_ci_hi',  'Behavioral: verbosity (1-5)',  'mean')
_bars(axes[1,2], df, 'bh_anxious',    'bh_anxious_ci_lo',    'bh_anxious_ci_hi',    'Behavioral: anxious_pessimism (1-5)', 'mean')
_bars(axes[1,3], df, 'bh_deference',  'bh_deference_ci_lo',  'bh_deference_ci_hi',  'Behavioral: deference (1-5)', 'mean')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'summary_bars.png', dpi=150, bbox_inches='tight')
plt.show()